# Django, Templates

## Introduction to Django Templates
---

In this lesson, you'll learn how to use **Django's template system** to generate HTML.

Templates are where your data meets the user interface - they combine HTML with dynamic content from your views.

1. [Template Syntax](#Template-Syntax),
    - [Variables](#Variables),
    - [Tags](#Tags),
    - [Filters](#Filters),
2. [Context and render()](#Context-and-render()),
    - [Passing Data to Templates](#Passing-Data-to-Templates),
    - [Context Processors](#Context-Processors),
3. [Template Inheritance](#Template-Inheritance),
    - [Base Templates](#Base-Templates),
    - [Blocks and Extends](#Blocks-and-Extends),
4. [Built-in Tags and Filters](#Built-in-Tags-and-Filters),
    - [Common Tags](#Common-Tags),
    - [Common Filters](#Common-Filters),
5. [🧠 EXERCISE 🧠, Create Templates for Bookstore](#🧠-EXERCISE-🧠,-Create-Templates-for-Bookstore).

<br>

## Template Syntax

---

Django templates use special syntax to mix HTML with dynamic content:

<br>

| Syntax | Purpose | Example |
|--------|---------|--------|
| `{{ }}` | Output variables | `{{ book.title }}` |
| `{% %}` | Template tags (logic) | `{% for book in books %}` |
| `{# #}` | Comments | `{# This is a comment #}` |

<br>

### Variables

---

Variables output values from the context. Use **dot notation** to access attributes, dictionary keys, list indices, or methods.

In [ ]:
# Template variable examples

template_variables = """
<!-- Simple variable -->
<h1>{{ title }}</h1>

<!-- Object attribute -->
<p>{{ book.title }}</p>

<!-- Dictionary key -->
<p>{{ user.name }}</p>

<!-- List index -->
<p>{{ items.0 }}</p>

<!-- Method call (no parentheses!) -->
<p>{{ name.upper }}</p>
"""
print(template_variables)

In [ ]:
# View providing context for the template above

from django.shortcuts import render

def example_view(request):
    context = {
        'title': 'Welcome to Bookstore',
        'book': {'title': 'Django Basics', 'author': 'John Doe'},
        'user': {'name': 'Alice', 'email': 'alice@example.com'},
        'items': ['First', 'Second', 'Third'],
        'name': 'hello world',
    }
    return render(request, 'example.html', context)

<br>

**Variable resolution order:**

When you write `{{ foo.bar }}`, Django tries in this order:

1. Dictionary lookup: `foo["bar"]`
2. Attribute lookup: `foo.bar`
3. List index: `foo[bar]` (if bar is numeric)

<br>

### Tags

---

Tags provide **logic** in templates - loops, conditionals, includes, etc.

In [ ]:
# Common template tags

template_tags = """
<!-- For loop -->
{% for book in books %}
    <p>{{ book.title }}</p>
{% endfor %}

<!-- For loop with empty -->
{% for book in books %}
    <p>{{ book.title }}</p>
{% empty %}
    <p>No books available.</p>
{% endfor %}

<!-- If/elif/else -->
{% if user.is_authenticated %}
    <p>Welcome, {{ user.username }}!</p>
{% elif user.is_guest %}
    <p>Welcome, guest!</p>
{% else %}
    <p>Please log in.</p>
{% endif %}

<!-- URL tag -->
<a href="{% url 'catalog:book_detail' pk=book.pk %}">View Book</a>
"""
print(template_tags)

<br>

### Filters

---

Filters **transform** variable values. Use the pipe `|` symbol.

In [ ]:
# Template filter examples

template_filters = """
<!-- Lowercase -->
{{ title|lower }}

<!-- Uppercase -->
{{ title|upper }}

<!-- Title case -->
{{ name|title }}

<!-- Default value -->
{{ description|default:"No description" }}

<!-- Truncate -->
{{ long_text|truncatewords:20 }}

<!-- Date formatting -->
{{ published_date|date:"F j, Y" }}

<!-- Length -->
{{ items|length }} items

<!-- Chaining filters -->
{{ name|lower|truncatewords:5 }}
"""
print(template_filters)

<br>

## Context and render()

---

### Passing Data to Templates

---

The **context** is a dictionary of data passed from the view to the template.

In [ ]:
# Example: Passing context to template

from django.shortcuts import render

def book_list(request):
    """Display list of books."""
    
    books = [
        {'id': 1, 'title': 'Django Basics', 'price': 29.99, 'in_stock': True},
        {'id': 2, 'title': 'Advanced Django', 'price': 49.99, 'in_stock': False},
        {'id': 3, 'title': 'Django REST', 'price': 39.99, 'in_stock': True},
    ]
    
    # Context dictionary - keys become variable names in template
    context = {
        'books': books,
        'page_title': 'Our Books',
        'show_prices': True,
    }
    
    return render(request, 'catalog/book_list.html', context)

In [ ]:
# The template: catalog/book_list.html

book_list_template = """
<!DOCTYPE html>
<html>
<head>
    <title>{{ page_title }}</title>
</head>
<body>
    <h1>{{ page_title }}</h1>
    
    {% for book in books %}
        <div class="book">
            <h2>{{ book.title }}</h2>
            
            {% if show_prices %}
                <p class="price">${{ book.price }}</p>
            {% endif %}
            
            {% if book.in_stock %}
                <span class="badge">In Stock</span>
            {% else %}
                <span class="badge sold-out">Sold Out</span>
            {% endif %}
        </div>
    {% empty %}
        <p>No books available.</p>
    {% endfor %}
</body>
</html>
"""
print(book_list_template)

<br>

### Context Processors

---

**Context processors** automatically add variables to every template context.

Django includes several by default:

| Processor | Provides |
|-----------|----------|
| `request` | The `request` object |
| `auth` | `user` and `perms` objects |
| `messages` | Flash messages |
| `debug` | Debug info (when DEBUG=True) |

In [ ]:
# Using built-in context variables in templates

template_context_vars = """
<!-- User from auth context processor -->
{% if user.is_authenticated %}
    <p>Logged in as: {{ user.username }}</p>
{% else %}
    <a href="{% url 'login' %}">Log in</a>
{% endif %}

<!-- Request from request context processor -->
<p>Current path: {{ request.path }}</p>

<!-- Messages from messages context processor -->
{% for message in messages %}
    <div class="alert alert-{{ message.tags }}">
        {{ message }}
    </div>
{% endfor %}
"""
print(template_context_vars)

<br>

## Template Inheritance

---

### Base Templates

---

Template inheritance lets you define a **base template** with common HTML structure, then **extend** it in child templates.

This follows the **DRY principle** (Don't Repeat Yourself).

In [ ]:
# Base template: templates/base.html

base_template = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{% block title %}Bookstore{% endblock %}</title>
    
    <!-- Common CSS -->
    <link rel="stylesheet" href="{% static 'css/style.css' %}">
    
    <!-- Page-specific CSS -->
    {% block extra_css %}{% endblock %}
</head>
<body>
    <header>
        <nav>
            <a href="{% url 'home' %}">Home</a>
            <a href="{% url 'catalog:book_list' %}">Books</a>
            <a href="{% url 'about' %}">About</a>
        </nav>
    </header>
    
    <main>
        {% block content %}
        <!-- Child templates fill this in -->
        {% endblock %}
    </main>
    
    <footer>
        <p>&copy; 2024 Bookstore</p>
    </footer>
    
    <!-- Common JS -->
    <script src="{% static 'js/main.js' %}"></script>
    
    <!-- Page-specific JS -->
    {% block extra_js %}{% endblock %}
</body>
</html>
"""
print(base_template)

<br>

### Blocks and Extends

---

Child templates use `{% extends %}` to inherit from a base template and `{% block %}` to override sections.

In [ ]:
# Child template: templates/catalog/book_list.html

child_template = """
{% extends 'base.html' %}

{% block title %}Books - Bookstore{% endblock %}

{% block content %}
<h1>Our Books</h1>

<div class="book-grid">
    {% for book in books %}
        <div class="book-card">
            <h2>{{ book.title }}</h2>
            <p class="author">{{ book.author }}</p>
            <p class="price">${{ book.price }}</p>
            <a href="{% url 'catalog:book_detail' pk=book.id %}" class="btn">
                View Details
            </a>
        </div>
    {% empty %}
        <p>No books found.</p>
    {% endfor %}
</div>
{% endblock %}
"""
print(child_template)

In [ ]:
# Another child template: templates/catalog/book_detail.html

detail_template = """
{% extends 'base.html' %}

{% block title %}{{ book.title }} - Bookstore{% endblock %}

{% block extra_css %}
<style>
    .book-detail { max-width: 800px; margin: 0 auto; }
</style>
{% endblock %}

{% block content %}
<article class="book-detail">
    <h1>{{ book.title }}</h1>
    <p class="author">By {{ book.author }}</p>
    <p class="description">{{ book.description|default:"No description available." }}</p>
    <p class="price">${{ book.price }}</p>
    
    <a href="{% url 'catalog:book_list' %}">← Back to Books</a>
</article>
{% endblock %}
"""
print(detail_template)

<br>

**Using `{{ block.super }}`** to include parent content:

In [ ]:
# Example: Extending parent block content

block_super_example = """
{% extends 'base.html' %}

{% block title %}{{ block.super }} | Books{% endblock %}
{# Result: "Bookstore | Books" #}

{% block extra_css %}
{{ block.super }}
<link rel="stylesheet" href="{% static 'css/books.css' %}">
{% endblock %}
"""
print(block_super_example)

<br>

## Built-in Tags and Filters

---

### Common Tags

---

In [ ]:
# for loop with forloop variables

for_loop_example = """
{% for book in books %}
    <p>
        {{ forloop.counter }}.     {# 1, 2, 3, ... #}
        {{ forloop.counter0 }}     {# 0, 1, 2, ... #}
        {{ book.title }}
        
        {% if forloop.first %}(First!){% endif %}
        {% if forloop.last %}(Last!){% endif %}
    </p>
{% endfor %}
"""
print(for_loop_example)

In [ ]:
# if tag with operators

if_examples = """
{% if books|length > 0 %}
    <p>We have {{ books|length }} books.</p>
{% endif %}

{% if user.is_staff and user.is_active %}
    <a href="/admin/">Admin Panel</a>
{% endif %}

{% if book.price >= 50 %}
    <span class="premium">Premium Book</span>
{% elif book.price >= 30 %}
    <span class="standard">Standard</span>
{% else %}
    <span class="budget">Budget</span>
{% endif %}

{% if "django" in book.title|lower %}
    <span class="django-book">Django Related</span>
{% endif %}
"""
print(if_examples)

In [ ]:
# url tag

url_examples = """
<!-- Simple URL -->
<a href="{% url 'home' %}">Home</a>

<!-- URL with argument -->
<a href="{% url 'catalog:book_detail' pk=book.pk %}">{{ book.title }}</a>

<!-- URL with multiple arguments -->
<a href="{% url 'archive' year=2024 month=3 %}">March 2024</a>

<!-- Store URL in variable -->
{% url 'catalog:book_list' as book_list_url %}
<a href="{{ book_list_url }}">All Books</a>
"""
print(url_examples)

In [ ]:
# include tag - include another template

include_examples = """
<!-- Simple include -->
{% include 'partials/header.html' %}

<!-- Include with extra context -->
{% include 'partials/book_card.html' with book=featured_book show_price=True %}

<!-- Include with only the specified context -->
{% include 'partials/book_card.html' with book=book only %}
"""
print(include_examples)

In [ ]:
# static tag - for static files

static_examples = """
{% load static %}

<link rel="stylesheet" href="{% static 'css/style.css' %}">
<script src="{% static 'js/app.js' %}"></script>
<img src="{% static 'images/logo.png' %}" alt="Logo">
"""
print(static_examples)

<br>

### Common Filters

---

In [ ]:
# String filters

string_filters = """
{{ name|lower }}                {# lowercase #}
{{ name|upper }}                {# UPPERCASE #}
{{ name|title }}                {# Title Case #}
{{ name|capfirst }}             {# First letter capitalized #}
{{ text|truncatewords:20 }}     {# Truncate to 20 words #}
{{ text|truncatechars:100 }}    {# Truncate to 100 characters #}
{{ text|linebreaks }}           {# Convert newlines to <p> and <br> #}
{{ text|linebreaksbr }}         {# Convert newlines to <br> #}
{{ text|striptags }}            {# Remove HTML tags #}
{{ text|slugify }}              {# URL-safe slug #}
"""
print(string_filters)

In [ ]:
# Date and number filters

date_number_filters = """
<!-- Date formatting -->
{{ published|date:"F j, Y" }}        {# January 15, 2024 #}
{{ published|date:"Y-m-d" }}         {# 2024-01-15 #}
{{ published|date:"SHORT_DATE_FORMAT" }}
{{ published|timesince }}            {# 3 days ago #}

<!-- Number formatting -->
{{ price|floatformat:2 }}            {# 29.99 #}
{{ count|add:10 }}                   {# Add 10 to count #}
{{ items|length }}                   {# List length #}
{{ value|divisibleby:3 }}            {# True if divisible by 3 #}
"""
print(date_number_filters)

In [ ]:
# Default and conditional filters

default_filters = """
<!-- Default values -->
{{ description|default:"No description" }}
{{ value|default_if_none:"N/A" }}

<!-- Boolean displays -->
{{ is_active|yesno:"Active,Inactive,Unknown" }}

<!-- Pluralize -->
{{ count }} book{{ count|pluralize }}
{{ count }} categor{{ count|pluralize:"y,ies" }}
"""
print(default_filters)

In [ ]:
# List filters

list_filters = """
{{ items|first }}                {# First item #}
{{ items|last }}                 {# Last item #}
{{ items|length }}               {# Number of items #}
{{ items|join:", " }}            {# Join with separator #}
{{ items|slice:":5" }}           {# First 5 items #}
{{ items|random }}               {# Random item #}
"""
print(list_filters)

<br>

## Template Organization

---

**Recommended directory structure:**

```
project/
├── templates/                  # Project-wide templates
│   ├── base.html               # Base template
│   ├── partials/               # Reusable components
│   │   ├── header.html
│   │   ├── footer.html
│   │   └── pagination.html
│   └── registration/           # Auth templates
│       ├── login.html
│       └── logout.html
├── catalog/
│   └── templates/
│       └── catalog/            # Namespaced by app name
│           ├── book_list.html
│           └── book_detail.html
└── accounts/
    └── templates/
        └── accounts/
            └── profile.html
```

<br>

**Configure in settings.py:**

In [ ]:
# settings.py template configuration

from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent.parent

TEMPLATES = [
    {
        'BACKEND': 'django.template.backends.django.DjangoTemplates',
        'DIRS': [
            BASE_DIR / 'templates',  # Project-wide templates
        ],
        'APP_DIRS': True,  # Look in app/templates/ directories
        'OPTIONS': {
            'context_processors': [
                'django.template.context_processors.debug',
                'django.template.context_processors.request',
                'django.contrib.auth.context_processors.auth',
                'django.contrib.messages.context_processors.messages',
            ],
        },
    },
]

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **Variables** | `{{ variable }}` - output values |
| **Tags** | `{% tag %}` - logic (for, if, url, etc.) |
| **Filters** | `{{ value\|filter }}` - transform values |
| **Context** | Dictionary passed from view to template |
| **render()** | `render(request, 'template.html', context)` |
| **Inheritance** | `{% extends 'base.html' %}` + `{% block %}` |
| **Include** | `{% include 'partial.html' %}` for reuse |
| **Static** | `{% load static %}` + `{% static 'file.css' %}` |

<br>

### 🧠 EXERCISE 🧠, Create Templates for Bookstore

---

Create a template structure for the bookstore project:

1. Create `templates/base.html` with:
   - Basic HTML structure
   - Navigation with links to home, books, about
   - Blocks: `title`, `content`, `extra_css`
2. Create `catalog/templates/catalog/book_list.html` that:
   - Extends base.html
   - Shows a list of books with title, author, price
   - Links each book to its detail page
3. Create `catalog/templates/catalog/book_detail.html` that:
   - Extends base.html
   - Shows full book information
   - Has a "Back to list" link
4. Update views to use `render()` with these templates
5. Test in browser

<details>
    <summary>▶️ Solution</summary>
    
**1. Create `templates/base.html`:**

```html
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{% block title %}Bookstore{% endblock %}</title>
    {% block extra_css %}{% endblock %}
</head>
<body>
    <nav>
        <a href="{% url 'home' %}">Home</a> |
        <a href="{% url 'catalog:book_list' %}">Books</a> |
        <a href="{% url 'about' %}">About</a>
    </nav>
    <hr>
    <main>
        {% block content %}{% endblock %}
    </main>
    <hr>
    <footer>
        <p>&copy; 2024 Bookstore</p>
    </footer>
</body>
</html>
```

**2. Create `catalog/templates/catalog/book_list.html`:**

```html
{% extends 'base.html' %}

{% block title %}Books - Bookstore{% endblock %}

{% block content %}
<h1>Our Books</h1>

{% for book in books %}
    <div style="margin-bottom: 20px; padding: 10px; border: 1px solid #ccc;">
        <h2>{{ book.title }}</h2>
        <p>Author: {{ book.author }}</p>
        <p>Price: ${{ book.price }}</p>
        <a href="{% url 'catalog:book_detail' pk=book.id %}">View Details</a>
    </div>
{% empty %}
    <p>No books available.</p>
{% endfor %}
{% endblock %}
```

**3. Create `catalog/templates/catalog/book_detail.html`:**

```html
{% extends 'base.html' %}

{% block title %}{{ book.title }} - Bookstore{% endblock %}

{% block content %}
<h1>{{ book.title }}</h1>
<p><strong>Author:</strong> {{ book.author }}</p>
<p><strong>Price:</strong> ${{ book.price }}</p>
<p><strong>Description:</strong> {{ book.description|default:"No description available." }}</p>

<p><a href="{% url 'catalog:book_list' %}">← Back to Books</a></p>
{% endblock %}
```

**4. Update `catalog/views.py`:**

```python
from django.shortcuts import render
from django.http import Http404

BOOKS = [
    {'id': 1, 'title': 'Django for Beginners', 'author': 'William Vincent', 'price': 29.99},
    {'id': 2, 'title': 'Two Scoops of Django', 'author': 'Daniel Feldroy', 'price': 49.99},
    {'id': 3, 'title': 'Django for Professionals', 'author': 'William Vincent', 'price': 39.99},
]

def book_list(request):
    return render(request, 'catalog/book_list.html', {'books': BOOKS})

def book_detail(request, pk: int):
    book = next((b for b in BOOKS if b['id'] == pk), None)
    if book is None:
        raise Http404("Book not found")
    return render(request, 'catalog/book_detail.html', {'book': book})
```

**5. Update `settings.py` to include templates dir:**

```python
TEMPLATES = [
    {
        'DIRS': [BASE_DIR / 'templates'],
        # ... rest of settings
    },
]
```

**Create directories:**

```bash
mkdir -p templates
mkdir -p catalog/templates/catalog
```
</details>

<br>

### 🧠 EXERCISE 🧠, Add Template Filters and Conditions

---

Enhance the book list template:

1. Add `in_stock` field to books (True/False)
2. Show "In Stock" or "Out of Stock" badge based on availability
3. Add `published_date` and display it formatted as "Month Day, Year"
4. Show book count: "Showing X book(s)"
5. Highlight books over $40 with a "Premium" label

<details>
    <summary>▶️ Solution</summary>
    
**1. Update BOOKS in `catalog/views.py`:**

```python
from datetime import date

BOOKS = [
    {
        'id': 1,
        'title': 'Django for Beginners',
        'author': 'William Vincent',
        'price': 29.99,
        'in_stock': True,
        'published_date': date(2023, 5, 15),
    },
    {
        'id': 2,
        'title': 'Two Scoops of Django',
        'author': 'Daniel Feldroy',
        'price': 49.99,
        'in_stock': False,
        'published_date': date(2022, 10, 1),
    },
    {
        'id': 3,
        'title': 'Django for Professionals',
        'author': 'William Vincent',
        'price': 39.99,
        'in_stock': True,
        'published_date': date(2024, 1, 20),
    },
]
```

**2-5. Update `catalog/templates/catalog/book_list.html`:**

```html
{% extends 'base.html' %}

{% block title %}Books - Bookstore{% endblock %}

{% block content %}
<h1>Our Books</h1>
<p>Showing {{ books|length }} book{{ books|length|pluralize }}.</p>

{% for book in books %}
    <div style="margin-bottom: 20px; padding: 10px; border: 1px solid #ccc;">
        <h2>
            {{ book.title }}
            {% if book.price >= 40 %}
                <span style="color: gold;">⭐ Premium</span>
            {% endif %}
        </h2>
        <p>Author: {{ book.author }}</p>
        <p>Price: ${{ book.price }}</p>
        <p>Published: {{ book.published_date|date:"F j, Y" }}</p>
        <p>
            {% if book.in_stock %}
                <span style="color: green;">✓ In Stock</span>
            {% else %}
                <span style="color: red;">✗ Out of Stock</span>
            {% endif %}
        </p>
        <a href="{% url 'catalog:book_detail' pk=book.id %}">View Details</a>
    </div>
{% empty %}
    <p>No books available.</p>
{% endfor %}
{% endblock %}
```
</details>

---